---
title: "Untitled"
output: html_document
date: "2025-10-06"
---



In [ ]:
knitr::opts_chunk$set(echo = TRUE)




## Load R libraries.



In [1]:
library(ggplot2)
library(ggrepel)
library(ggpubr)
library(stringr)
library(MASS)
library(RColorBrewer)
library(viridis)
library(ggpointdensity)
library(dplyr)
library(data.table)
library(readxl)

theme_set(theme_classic())

personal_path = "/fh/working/sun_w/sshen/MorPhiC"
path = "/fh/fast/sun_w/MorPhiC/data/MorPhiC_exchange_experiment/"
dat_path = file.path(path, "DRACC-processed/NW_exchange_experiment_DRACC_processed_March2026/Tables")

read_header <- function(file_name, sheet){
  meta_header = fread(file_name)
  meta_header = as.character(meta_header)
  meta_header = gsub("_cell_line", "", meta_header, fixed = TRUE)
  meta_header = gsub("differentiated_product.", "", meta_header, fixed = TRUE)
  meta_header = gsub(".text", "", meta_header, fixed = TRUE)
  meta_header
}


Loading required package: viridisLite


Attaching package: ‘dplyr’


The following object is masked from ‘package:MASS’:

    select


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last




In [2]:
# Load required libraries
library(data.table)
library(stringr)
library(dplyr)

meta_file = paste0(personal_path, "/NW_meta_data.csv")

meta_df = fread(meta_file)
meta_df = as.data.frame(meta_df)
dim(meta_df)

# 2. Transform the data to match the TSV format
new_meta <- meta_df %>%
  mutate(
    # Get the original ID
    Orig_ID = `DIFFERENTIATED PRODUCT ID (Required)`,
    
    # Extract Replicate: the last sequence of digits at the end of the string
    Replicate = str_extract(Orig_ID, "\\d+$"),
    
    # Extract Dcondition (Base name): Remove "EX_" prefix and trailing numbers/spaces
    Dcondition = str_replace(Orig_ID, "^EX_", ""),
    Dcondition = str_replace(Dcondition, "[ _]?\\d+$", ""),
    
    # Construct Name
    Name = NAMES,
    
    # Determine Center based on keywords in the ID
    Center = case_when(
      str_detect(Orig_ID, "NW|OSTIR") ~ "NorthWestern",
      str_detect(Orig_ID, "UCSF") ~ "UCSF",
      str_detect(Orig_ID, "JAX") ~ "JAX",
      str_detect(Orig_ID, "MSK") ~ "MSK",
      str_detect(Orig_ID, "KOLF") ~ "Parent",
      TRUE ~ "Unknown"
    ),
    
    # Handle NA values for treatments and controls
    Treat = ifelse(is.na(`TREATMENT/CONDITION`), "None", `TREATMENT/CONDITION`),
    WT = ifelse(is.na(`WT/CONTROL STATUS`), "None", `WT/CONTROL STATUS`),
    
    # Construct Condition
    Condition = paste0(WT, " plus ", Treat)
  ) %>%
  # Fix phrasing if there was no treatment
  mutate(Condition = str_replace(Condition, " plus None", " none")) %>%
  # Keep only the 5 columns that match the UCSF_meta_data.tsv format
  select(Name, Center, Replicate, Condition, Dcondition)

new_meta$ko_gene = "WT"
new_meta$ko_gene[new_meta$Dcondition == "NW_C39_AUXIN"] = "KO"
new_meta$ko_gene[new_meta$Dcondition == "NW_C14_AUXIN"] = "KO"
new_meta$ko_gene[new_meta$Dcondition == "JAX_CLONE_DO4"] = "KO"
new_meta$ko_gene[new_meta$Dcondition == "JAX_CLONE_AD2"] = "KO"
new_meta$ko_gene[new_meta$Dcondition == "MSK_CLONE_05"] = "KO"
new_meta$ko_gene[new_meta$Dcondition == "MSK_CLONE_07"] = "KO"
new_meta$ko_gene[new_meta$Dcondition == "UCSF_CLONE_D1_DOX"] = "KO"
new_meta$ko_gene[new_meta$Dcondition == "UCSF_CLONE_B1 DOX"] = "KO"


[1] 57 11


## Check count data



In [3]:
cts = fread(file.path(dat_path, "genesCounts.csv"), data.table = FALSE)
cts = cts[, -1]
colnames(cts) <- gsub("_S[0-9]+_L[0-9]+$", "", colnames(cts))

new_meta = new_meta[match(colnames(cts), new_meta$Name), ]
fwrite(new_meta, file = file.path(personal_path, "NW_meta_data.tsv"), sep="\t")



In [4]:
gc()
sessionInfo()


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1264306,67.6,2511037,134.2,2511037,134.2
Vcells,3781494,28.9,8388608,64.0,5297527,40.5


R version 4.5.3 (2026-03-11)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 24.04.4 LTS

Matrix products: default
BLAS/LAPACK: /home/sshen2/.conda/envs/r/lib/libopenblasp-r0.3.33.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: America/Los_Angeles
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] readxl_1.5.0         data.table_1.18.4    dplyr_1.2.1         
 [4] ggpointdensity_0.2.1 viridis_0.6.5        viridisLite_0.4.3   
 [7] RColorBrewer_1.1-3   MASS_7.3-65          stringr_1.6.0       
[10] ggpubr_1.0.0         ggrepel_0.9.8        ggplot2_4.0.3       

loaded via a namespac